## 1. Setup

First, we load the data, TransformerLens model and other global variables

In [15]:
import os
from pathlib import Path

ROOT = Path("/Users/joflesan/Documents/University/Masters/Year 2/SEM 1/Semester Project/decompiling_transformers")
os.chdir(ROOT / "src")  # Go to root
os.environ['TRANSFORMERLENS_ALLOW_MPS'] = "1"

from functools import partial
from itertools import product
from typing import Callable, Literal, Tuple

import circuitsvis as cv
import einops
import random
import numpy as np
import plotly.express as px
import torch as t
from IPython.display import HTML, display
from jaxtyping import Bool, Float, Int
from rich import print as rprint
from rich.table import Column, Table
from torch import Tensor
from tqdm.notebook import tqdm
from transformer_lens import ActivationCache, HookedTransformer, utils
from transformer_lens.model_bridge import TransformerBridge
from transformer_lens.components import MLP, Embed, LayerNorm, Unembed
from transformer_lens.hook_points import HookPoint

# Local imports
from mechanistic.utilities.mechinterp_utils import residual_stack_to_logit, topk_of_Nd_tensor
from mechanistic.utilities.metrics import get_logit_diff
from mechanistic.utilities.mechinterp_viz import line, imshow
from data.CustomTokenizer import CustomTokenizer
from data.CountDataset import CountCorruption, CountDataset

t.set_grad_enabled(False)
device = t.device(
    "mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu"
)

MAX_TEST_LENGTH = 150
MODEL_PATH = ROOT / "models/count-%2l4h256d4lr01drop"

In [7]:
# Initialize counting tokenizer and create dummy example
vocab = [str(i) for i in range(MAX_TEST_LENGTH)]
tokenizer = CustomTokenizer(vocab)
example_prompt = "<bos> 2 5 <sep> 2 3"

tokens = tokenizer(example_prompt)
max_offset = MAX_TEST_LENGTH - len(tokens)
offset = random.randint(0, max_offset) if max_offset >= 0 else 0
example_pos_ids = list(range(offset, offset + len(tokens['input_ids'][0])))


# Initialize the model and load it using TransformerLens
model = TransformerBridge.boot_transformers(
    MODEL_PATH
)
model.enable_compatibility_mode(
    fold_ln=True,
    center_unembed=True,
    center_writing_weights=True,
    refactor_factored_attn_matrices=True
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

Loading weights: 100%|██████████| 28/28 [00:00<00:00, 6750.23it/s]


In [8]:
prompts = [
    "<bos> 2 5 <sep>",
    "<bos> 1 8 <sep>",
    "<bos> 8 12 <sep>",
    "<bos> 11 15 <sep>",
]

# Define the answers for each prompt, in the form (correct, incorrect)
answers = [2, 1, 8, 11]
answers = [(ans, f"!={ans}") for ans in answers]

# Define the answer tokens (same shape as the answers)
answer_tokens = t.tensor([tokenizer(str(gt))['input_ids'].item() for gt, _ in answers])

rprint(answer_tokens)

table = Table("Prompt", "Correct", "Incorrect", title="Prompts & Answers:")

for prompt, answer in zip(prompts, answers):
    table.add_row(prompt, repr(answer[0]), repr(answer[1]))

rprint(table)

tensor([ 2,  1,  8, 11])

            Prompts & Answers:             
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┓
┃ Prompt            ┃ Correct ┃ Incorrect ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━┩
│ <bos> 2 5 <sep>   │ 2       │ '!=2'     │
│ <bos> 1 8 <sep>   │ 1       │ '!=1'     │
│ <bos> 8 12 <sep>  │ 8       │ '!=8'     │
│ <bos> 11 15 <sep> │ 11      │ '!=11'    │
└───────────────────┴─────────┴───────────┘

In [9]:
# Tokenize the prompts
tokens = tokenizer(prompts)['input_ids']
position_ids = t.empty_like(tokens)
position_ids[:, :] = t.arange(0, tokens.size(1))

# Compute logits and cache for the first prompt
logits, cache = model.run_with_cache(
    tokens.to(device),
    position_ids=position_ids.to(device)
)

## 2. Logit Attribution

In this section, we explore which components in the network contribute the most towards the logit difference of interest

In [10]:
answer_residual_directions = model.tokens_to_residual_directions(answer_tokens)  # [batch d_model]
print("Answer residual directions shape:", answer_residual_directions.shape)

Answer residual directions shape: torch.Size([4, 256])


We can verify that the residual direction computed is as expected by applying the LayerNorm scaling directly to the post residual cached activations for the last position

In [11]:
# Cache syntax: resid_post is the residual stream at the end of the layer, -1 gets the final layer.
# The general syntax is [activation_name, layer_index, sub_layer_type].
final_residual_stream: Float[Tensor, "batch seq d_model"] = cache["resid_post", -1]  # Get final layer post-residual cache
print(f"Final residual stream shape: {final_residual_stream.shape}")
final_token_residual_stream: Float[Tensor, "batch d_model"] = final_residual_stream[:, -1, :]  # Get final token predictions

# Apply LayerNorm scaling (to just the final sequence position)
# pos_slice is the subset of the positions we take - here the final token of each prompt
scaled_final_token_residual_stream = cache.apply_ln_to_stack(
    final_token_residual_stream, layer=-1, pos_slice=-1
)

# Get unembedding directions for the target logits
target_logit_directions = model.W_U[:, answer_tokens].T  # [batch, d_model]

# Compute logits via dot product
reconstructed_logits = einops.einsum(
    scaled_final_token_residual_stream,
    target_logit_directions,
    "batch d_model, batch d_model -> batch"
) + model.b_U[answer_tokens]  # NOTE: because we are not computing logitDiff, bias matters!

# Compare to actual logits
actual_logits = logits[
    t.arange(logits.size(0)),
    -1,
    answer_tokens
]

rprint(reconstructed_logits)
rprint(actual_logits)
t.testing.assert_close(reconstructed_logits, actual_logits)

Final residual stream shape: torch.Size([4, 4, 256])


tensor([11.3140, 11.6557, 11.7908, 11.3216], device='mps:0')

tensor([11.3140, 11.6557, 11.7908, 11.3216], device='mps:0')

In [12]:
# LogitLens Attribution by component
target_logit_directions = model.W_U[:, answer_tokens].T

accumulated_residual, labels = cache.accumulated_resid(
    layer=-1,
    incl_mid=True,
    pos_slice=-1,
    return_labels=True,
)

logit_lens_logits: Float[Tensor, "component"] = residual_stack_to_logit(
    accumulated_residual,
    cache,
    target_logit_directions,
)

line(
    logit_lens_logits,
    hovermode="x unified",
    title="Logit Contribution From Accumulated Residual Stream",
    labels={"x": "Layer", "y": "Logit Contribution"},
    xaxis_tickvals=labels,
    width=800,
)

From this we see that the counting logic can mostly be attributed to the MLP in layer 0 (first jump) and the MLP in layer 1 (second jump). 

**TODO**: Check whether this is consistent with the decompiling transformers results

In [122]:
per_layer_residual, labels = cache.decompose_resid(layer=-1, pos_slice=-1, return_labels=True)
per_layer_logits = residual_stack_to_logit(per_layer_residual, cache, target_logit_directions,)

line(
    per_layer_logits,
    hovermode="x unified",
    title="Logit Contribution From Each Layer",
    labels={"x": "Layer", "y": "Logit Contribution"},
    xaxis_tickvals=labels,
    width=800,
)

Again, this is confirmed by investigating the layer contributions directly. It makes sense that the MLPs are responsible for this behaviour as this involves processing the tokens, not just copying them around the residual stream (which would be the job of the Attention Heads). However, we can verify this by looking at the attributions for the Attention Heads

### Attention Head Attribution

To verify whether we can attribute any logit contributions to any of the attention heads, we can inspect these as well...

In [123]:
per_head_residual, labels = cache.stack_head_results(layer=-1, pos_slice=-1, return_labels=True)
per_head_residual = einops.rearrange(
    per_head_residual, "(layer head) ... -> layer head ...", layer=model.cfg.n_layers
)
per_head_logit_diffs = residual_stack_to_logit(per_head_residual, cache, target_logit_directions)

imshow(
    per_head_logit_diffs,
    labels={"x": "Head", "y": "Layer"},
    title="Logit Difference From Each Head",
    width=600,
)

Interestingly, the second layer's Attention Heads are influencing the final logit somewhat (although still comparatively weakly) and we can see some destructive destructive behaviour from L1H0 and some constructive behaviour from L1H3. Generally, the later heads in this layer seem to contribute more and more positively, but again this could simply be noise

In [124]:
k = 3

for head_type in ["Positive", "Negative"]:
    # Get the heads with largest (or smallest) contribution to the logit difference
    top_heads = topk_of_Nd_tensor(
        per_head_logit_diffs * (1 if head_type == "Positive" else -1), k
    )

    # Get all their attention patterns
    attn_patterns_for_important_heads: Float[Tensor, "head q k"] = t.stack(
        [cache["pattern", layer][:, head][0] for layer, head in top_heads]
    )

    # Display results
    safe_tokens = tokenizer.convert_ids_to_tokens(tokens[0])
    safe_tokens[0] = "BOS"
    safe_tokens[3] = "SEP"
    display(HTML(f"<h2>Top {k} {head_type} Logit Attribution Heads</h2>"))
    display(
        cv.attention.attention_patterns(
            attention=attn_patterns_for_important_heads,
            tokens=safe_tokens,
            attention_head_names=[f"{layer}.{head}" for layer, head in top_heads],
        )
    )

## Activation Patching

In this section, we look at running activation patching in order to detect important components for this particular behaviour. This will complement our logit attribution results

### Creating a Metric

We start by defining a metric that is 0 when denoising does not change the behaviour of the model and 1 when it perfectly recovers the clean signal

In [3]:
def tokenize_prompt(prompt: str) -> Tuple[Float[Tensor, "1 seq"], Float[Tensor, "1 seq"]]:
    tokens = tokenizer(prompt)['input_ids']
    max_offset = MAX_TEST_LENGTH - len(tokens)
    offset = random.randint(0, max_offset) if max_offset >= 0 else 0
    clean_pos_ids = list(range(offset, offset + len(tokens[0])))
    position_ids = t.tensor(clean_pos_ids).unsqueeze(0)
    
    return tokens, position_ids

def decode_greedy(model, prompt_tokens, max_new_tokens=10):
    """Greedily decode from the model given a prompt."""
    tokens = prompt_tokens.clone().to(device)
    for _ in range(max_new_tokens):
        logits = model(tokens)
        next_token = logits[0, -1, :].argmax(dim=-1, keepdim=True).unsqueeze(0).to(device)
        tokens = t.cat([tokens, next_token], dim=-1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
        
    return tokens

In [3]:
count_dataset = CountDataset(
    tokenizer=tokenizer,
    length_range=[0, 150],
    max_test_length=150
)

corruptedData = count_dataset.get_corrupted(CountCorruption.CHANGE_START)
clean_tokens = corruptedData.clean_tokens
clean_pos = corruptedData.clean_pos
corrupt_tokens = corruptedData.corrupted_tokens
corrupt_pos = corruptedData.corrupted_pos
answer_tokens = corruptedData.answer_tokens

In [76]:
# H0: there are components learning to copy from the second token to the first position
# predicted. In other words, they learn where to start counting
clean_prompt = "<bos> 1 5 <sep>"  # Target: 1
corrupt_prompt = "<bos> 4 5 <sep>"  # Target: 4

clean_tokens, clean_pos = tokenize_prompt(clean_prompt)
corrupt_tokens, corrupt_pos = tokenize_prompt(corrupt_prompt)
answer_tokens = t.cat([tokenizer("1")['input_ids'], tokenizer("4")['input_ids']], dim=1)

In [64]:
# Decode the model's outputs to see what the completions look like
clean_full = decode_greedy(model, clean_tokens)
corrupted_full = decode_greedy(model, corrupt_tokens)

print(f"Clean completion: {tokenizer.convert_ids_to_tokens(clean_full[0])}")
print(f"Corrupted completion: {tokenizer.convert_ids_to_tokens(corrupted_full[0])}")

# Check logit distribution at SEP to understand what is happening
clean_logits, _ = model.run_with_cache(clean_tokens.to(device))
top_k = clean_logits[0, 3, :].topk(5)
print("\nTop 5 predictions at SEP position:")
for val, idx in zip(top_k.values, top_k.indices):
    print(f"  {tokenizer.convert_ids_to_tokens([idx.item()])!r:10s}  logit={val:.2f}")

Clean completion: ['<bos>', '1', '5', '<sep>', '1', '2', '3', '4', '5', '<eos>']
Corrupted completion: ['<bos>', '4', '5', '<sep>', '4', '5', '<eos>']

Top 5 predictions at SEP position:
  ['1']       logit=11.58
  ['5']       logit=4.57
  ['0']       logit=4.55
  ['2']       logit=4.00
  ['9']       logit=2.91


In [65]:
clean_logits, clean_cache = model.run_with_cache(
    clean_tokens.to(device),
    position_ids=clean_pos.to(device)
)
corrupted_logits, corrupted_cache = model.run_with_cache(
    corrupt_tokens.to(device),
    position_ids=corrupt_pos.to(device)
)

# Sanity check -- model should predict 2 from clean, 7 from corrupted
print(f"Clean top prediction: {tokenizer.convert_ids_to_tokens([clean_logits[0, -1].argmax().item()])}")
print(f"Corrupt top prediction: {tokenizer.convert_ids_to_tokens([corrupted_logits[0, -1].argmax().item()])}")

Clean top prediction: ['1']
Corrupt top prediction: ['4']


In [78]:
#  Define the metric
#  Measure logit(clean_start) - logit(corrupted_start) at the SEP position
#
#  +1 -> component pushes the model towards the CLEAN answer (1)
#  -1 -> component pushes the model towards the CORRUPTED answer (4)
#  0 -> component is uninvolved in routing the start token

def logit_diff_metric(
    logits: Float[Tensor, "batch seq d_vocab"],
    answer_tokens: Int[Tensor, "batch 2"],
    corrupted_logit_diff: float,
    clean_logit_diff: float,
) -> Float[Tensor, "..."]:
    """
    Linear function of logit diff, calibrated so that it equals 0 when performance is same as on
    corrupted input, and 1 when performance is same as on clean input.
    """
    
    patched_logit_diff = get_logit_diff(logits, answer_tokens)
    return (patched_logit_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

clean_logit_diff = get_logit_diff(clean_logits, answer_tokens)
corrupted_logit_diff = get_logit_diff(corrupted_logits, answer_tokens)

print(f"Clean logit diff: {clean_logit_diff:.4f}")
print(f"Corrupted logit diff: {corrupted_logit_diff:.4f}")

Clean logit diff: 9.0457
Corrupted logit diff: -8.1389


In [81]:
from transformer_lens import patching

### 1. Residual Stream Patching

In [85]:
act_patch_block_every = patching.get_act_patch_block_every(
    model, corrupt_tokens.to(device), clean_cache, partial(
        logit_diff_metric, answer_tokens=answer_tokens, corrupted_logit_diff=corrupted_logit_diff,
        clean_logit_diff=clean_logit_diff
    )
)

100%|██████████| 8/8 [00:00<00:00, 88.43it/s]


In [91]:
labels = [f"{tok} {i}" for i, tok in enumerate(model.to_str_tokens(clean_tokens[0]))]
imshow(
    act_patch_block_every,
    x=labels,
    facet_col=0, # This argument tells plotly which dimension to split into separate plots
    facet_labels=["Residual Stream", "Attn Output", "MLP Output"], # Subtitles of separate plots
    title="Logit Difference From Patched Residual Stream",
    labels={"x": "Sequence Position", "y": "Layer"},
    width=1200,
)

**Interpretation**: information about the start token is present in the residual stream early on (from layer 0), and by layer 1 it is moved to the right position (position 3). Importantly, all information about this token is moved such that information in position 1 of the residual stream is no longer relevant.

As we suspect, there is an attention head writing the start token information into position 3's residual stream and this head lives in layer 0. This means a head in this layer attends back to position 1.

Both MLPs are involved, but importantly, they act on information in position 3 of the residual stream directly. They do not perform any copying or processing on position 1 information. This means the MLP is likely doing some embedding mapping to convert to the right logit prediction. They could also be performing a no-op?

### 2. Attention Head Patching

In [88]:
act_patch_attn_head_out_all_pos = patching.get_act_patch_attn_head_out_all_pos(
    model, corrupt_tokens.to(device), clean_cache, partial(
        logit_diff_metric, answer_tokens=answer_tokens, corrupted_logit_diff=corrupted_logit_diff,
        clean_logit_diff=clean_logit_diff
    )
)

imshow(
    act_patch_attn_head_out_all_pos,
    labels={"y": "Layer", "x": "Head"},
    title="attn_head_out Activation Patching (All Pos)",
    width=600
)

100%|██████████| 8/8 [00:00<00:00, 45.71it/s]


**Interpretation**: we again see that head 0.3 seems to be the main head involved in copying the information from position 1 to position 3 in the residual stream. Other heads however are still causally involved which could mean they are either backup mover heads or they are performing some other operation on the other positions that affects the metric. However, it is clear that attention heads in layer 1 are not causally involved in the copying procedure

### 3. Decomposed Attention Head Patching

In [90]:
act_patch_attn_head_all_pos_every = patching.get_act_patch_attn_head_all_pos_every(
    model, corrupt_tokens.to(device), clean_cache, partial(
        logit_diff_metric, answer_tokens=answer_tokens, corrupted_logit_diff=corrupted_logit_diff,
        clean_logit_diff=clean_logit_diff
    )
)

imshow(
    act_patch_attn_head_all_pos_every,
    facet_col=0,
    facet_labels=["Output", "Query", "Key", "Value", "Pattern"],
    title="Activation Patching Per Head (All Pos)",
    labels={"x": "Head", "y": "Layer"},
)

100%|██████████| 8/8 [00:00<00:00, 122.00it/s]


**Interpretation**: Interestingly, the heads are important not because of what they attend to, but rather because of what they copy. This means that the heads are definitely attending to a specific position in the sequence (pos=1) rather than to a learned offset or some other operation, but in this case they encounter the wrong value. It is still unclear what the roles of heads 0.0-0.2 is in the network, but I hypothesize these are simply backup heads, which we can test by looking at their attention patterns

### 4. Investigating Attention Heads

In order to determine what exactly the attention heads in layer 0 are doing, we can inspect their attention patterns directly

In [95]:
# Get the heads with largest value patching
# (we know from plot above that these are the 4 heads in layers 7 & 8)
k = 4
top_heads = topk_of_Nd_tensor(act_patch_attn_head_all_pos_every[3], k=k)

# Get all their attention patterns
attn_patterns_for_important_heads: Float[Tensor, "head q k"] = t.stack([
    cache["pattern", layer][:, head].mean(0)
        for layer, head in top_heads
])

# Display results
safe_tokens = tokenizer.convert_ids_to_tokens(clean_tokens[0])
safe_tokens[0] = "BOS"
safe_tokens[3] = "SEP"
display(HTML(f"<h2>Top {k} Logit Attribution Heads (from value-patching)</h2>"))
display(cv.attention.attention_patterns(
    attention = attn_patterns_for_important_heads,
    tokens = safe_tokens,
    attention_head_names = [f"{layer}.{head}" for layer, head in top_heads],
))

**Interpretation**: counter-intuitively, it seems that head 0.0 has the largest bias for attending to position 1 at SEP, whereas the other heads seem to be (semi-) uniformly attending to all previous positions. This means heads are computing a weighted average of the context which could itself carry information of the start token. This kind of behaviour possibly contradicts our hypothesis, suggesting there is no "copying" behaviour in the network, but rather some kind of recurrent computation, where context is aggregated and the MLPs learn to transform this context into the next prediction. From this, it makes sense to explore the role of the MLPs

In [110]:
# --------- VERIFY COPYING HEAD BY INSPECTING OV CIRCUIT ---------

layer, head = 0, 0 # The candidate copying head

W_V = model.W_V[layer, head]   # [d_model, d_head]
W_O = model.W_O[layer, head]   # [d_head, d_model]
W_OV = W_V @ W_O               # [d_model, d_model]

# Project onto the token embedding space
W_E = model.W_E                # [vocab, d_model]
OV_circuit = W_E @ W_OV @ W_E.T  # [vocab, vocab]

# If this is a copying head, OV_circuit should be
# approximately diagonal — token i maps to token i
import plotly.express as px
px.imshow(OV_circuit.detach().cpu().numpy(),
        title="OV Circuit for L0H0 — diagonal = copying")

**Interpretation**: this is strong evidence that the attention heads are actually involved in the increment operation themselves. In other words, each head learns to attend to the n - 1 position such that token j - 1 predicts token j. The effect is small but consistent among most heads. This also aligns with the findings from the decompiling transformers paper.

The question at this point is what exactly are the MLPs doing in this model. It seems like they might just be reinforcing the logit of the next token, but this information is already correctly predicted by the attention layers.

In [101]:
# ---------- INSPECT QK CIRCUIT TO UNDERSTAND ATTENTION BIAS -------------

W_Q = model.W_Q[layer, head]   # [d_model, d_head]
W_K = model.W_K[layer, head]   # [d_model, d_head]
W_QK = W_Q @ W_K.T             # [d_model, d_model]

# Project: what does SEP's query attend to across all token embeddings?
sep_embedding = model.W_E[tokenizer("<sep>")['input_ids'].item()]
qk_scores = sep_embedding @ W_QK @ model.W_E.T  # [vocab]

# High scores = tokens that SEP naturally attends to regardless of content
print(f"TOKENS THAT SEP ATTENDS TO REGARDLESS OF CONTEXT:")
print(tokenizer.convert_ids_to_tokens(qk_scores.topk(5).indices))

TOKENS THAT SEP ATTENDS TO REGARDLESS OF CONTEXT:
['1', '3', '0', '7', '6']


In [114]:
# Check what the residual stream at SEP predicts after each layer
for layer in range(model.cfg.n_layers):
    resid = clean_cache[f"blocks.{layer}.hook_resid_mid"]  # [batch, seq, d_model]
    logits = model.unembed(model.ln_final(resid))[0, 3]
    top = logits.topk(3)
    print(f"Layer {layer}: {tokenizer.convert_ids_to_tokens(top.indices.tolist())} {top.values.tolist()}")

Layer 0: ['<eos>', '137', '127'] [3.4499573707580566, 2.0510623455047607, 1.6668862104415894]
Layer 1: ['1', '0', '2'] [5.150368690490723, 3.821410894393921, 2.9490134716033936]


## Path Patching

In this section, we try to find a causally important path based on the components we suspect are involved in the first token prediction. This will allow us to argue for specific interactions between components instead of just isolating individual fine-grained behaviours

## Mean Ablations

Before trying to find minimal circuits, we can run some quick tests to check if the paths and heads identified are causally important.

## ACDC

Automatic Circuit Discovery using existing ACDC method

In [ ]:
from acdc.TLACDCExperiment import TLACDCExperiment

ImportError: cannot import name 'path_patch' from 'transformer_lens.patching' (/Users/joflesan/Documents/University/Masters/Year 2/SEM 1/Semester Project/decompiling_transformers/.venv/lib/python3.11/site-packages/transformer_lens/patching.py)

In [ ]:
path_patch(
    model,
    orig_input=corrupt_tokens,
    new_input=clean_tokens,
    sender_nodes=[("z", 0, 0)],
    receiver_nodes=[("q", 2, 4)],
    patching_metric=logit_diff_metric
)